In [2]:
# ============================================
# EXP25: ElasticNet with Spectral Binning
# ============================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline


# =========================
# Load data
# =========================

train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

target = "含水率"

ignore_cols = [
    "sample number",
    "species number",
    "樹種",
    target
]

spectral_cols = [c for c in train.columns if c not in ignore_cols]


X_train = train[spectral_cols]
y_train = train[target]

X_test = test[spectral_cols]


# =========================
# Spectral binning function
# =========================

def spectral_binning(df, bin_size=5):

    cols = df.columns
    binned = []

    for i in range(0, len(cols), bin_size):

        group = cols[i:i+bin_size]

        name = f"bin_{i}"

        binned_feature = df[group].mean(axis=1)

        binned.append(binned_feature.rename(name))

    return pd.concat(binned, axis=1)


# =========================
# Apply binning
# =========================

X_train_binned = spectral_binning(X_train, bin_size=5)
X_test_binned = spectral_binning(X_test, bin_size=5)


print("Original features:", X_train.shape[1])
print("Binned features:", X_train_binned.shape[1])


# =========================
# Model
# =========================

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.002,
        l1_ratio=0.9,
        max_iter=200000,
        random_state=42
    ))
])


# =========================
# Train
# =========================

model.fit(X_train_binned, y_train)


# =========================
# Predict
# =========================

preds = model.predict(X_test_binned)


# =========================
# Submission
# =========================

submission = pd.DataFrame({
    "sample number": test["sample number"],
    target: preds
})

file_name = "submissions/exp25_elasticnet_binned.csv"

submission.to_csv(file_name, index=False)

print("Saved:", file_name)

Original features: 1555
Binned features: 311


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.294e+04, tolerance: 3.242e+02
  model = cd_fast.enet_coordinate_descent(


OSError: Cannot save file into a non-existent directory: 'submissions'

In [3]:
# ============================================
# EXP25: ElasticNet with Spectral Binning
# ============================================

import os
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline


# =========================
# Ensure submissions folder exists
# =========================

os.makedirs("submissions", exist_ok=True)


# =========================
# Load data
# =========================

train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

target = "含水率"

ignore_cols = [
    "sample number",
    "species number",
    "樹種",
    target
]

spectral_cols = [c for c in train.columns if c not in ignore_cols]

X_train = train[spectral_cols]
y_train = train[target]

X_test = test[spectral_cols]


# =========================
# Spectral binning function
# =========================

def spectral_binning(df, bin_size=5):

    cols = df.columns
    binned_features = []

    for i in range(0, len(cols), bin_size):

        group = cols[i:i+bin_size]

        name = f"bin_{i}"

        feature = df[group].mean(axis=1)

        binned_features.append(feature.rename(name))

    return pd.concat(binned_features, axis=1)


# =========================
# Apply binning
# =========================

X_train_binned = spectral_binning(X_train, bin_size=5)
X_test_binned = spectral_binning(X_test, bin_size=5)

print("Original features:", X_train.shape[1])
print("Binned features:", X_train_binned.shape[1])


# =========================
# Model
# =========================

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", ElasticNet(
        alpha=0.005,
        l1_ratio=0.9,
        max_iter=300000,
        tol=1e-3,
        random_state=42
    ))
])


# =========================
# Train model
# =========================

model.fit(X_train_binned, y_train)


# =========================
# Predict
# =========================

preds = model.predict(X_test_binned)


# =========================
# Create submission
# =========================

submission = pd.DataFrame({
    "sample number": test["sample number"],
    target: preds
})


# =========================
# Save submission
# =========================

file_name = "submissions/exp25_elasticnet_binned.csv"

submission.to_csv(file_name, index=False)

print("Saved:", file_name)

Original features: 1555
Binned features: 311
Saved: submissions/exp25_elasticnet_binned.csv


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.733e+03, tolerance: 3.242e+03
  model = cd_fast.enet_coordinate_descent(


In [4]:
# ================================
# exp25_elasticnet_spectral_binning
# ElasticNet with spectral feature binning
# ================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error


# ================================
# Metric diagnostics tool
# ================================

def metric_diagnostics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(np.maximum(y_pred,0))
    rmsle = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    nrmse_mean = rmse / np.mean(y_true)
    nrmse_range = rmse / (np.max(y_true) - np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("RMSLE:", rmsle)
    print("NRMSE (mean):", nrmse_mean)
    print("NRMSE (range):", nrmse_range)


# ================================
# Spectral binning function
# ================================

def spectral_binning(df, bin_size=5):

    cols = df.columns
    binned = []

    for i in range(0, len(cols), bin_size):

        group = cols[i:i+bin_size]
        name = f"bin_{i}"

        feature = df[group].mean(axis=1)

        binned.append(feature.rename(name))

    return pd.concat(binned, axis=1)


# ================================
# Load data
# ================================

train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

target = "含水率"
id_col = "sample number"

spectral_cols = [
    c for c in train.columns
    if c not in ["sample number","species number","樹種","含水率"]
]

X = train[spectral_cols]
y = train[target]

X_test = test[spectral_cols]


# ================================
# Apply spectral binning
# ================================

X = spectral_binning(X, bin_size=5)
X_test = spectral_binning(X_test, bin_size=5)

print("Original features:", len(spectral_cols))
print("Binned features:", X.shape[1])


# ================================
# ElasticNet configurations
# ================================

elastic_configs = [
    (0.0005, 0.9),
    (0.001, 0.9),
    (0.002, 0.9),
    (0.001, 0.8)
]


# ================================
# KFold
# ================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))


# ================================
# Training loop
# ================================

for alpha, l1_ratio in elastic_configs:

    print(f"\nTraining ElasticNet alpha={alpha}, l1_ratio={l1_ratio}")

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=50000,
            random_state=42
        ))
    ])

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)

        pred_val = model.predict(X_val)

        oof[val_idx] = pred_val

        test_pred += model.predict(X_test) / kf.n_splits

    metric_diagnostics(y, oof)

    oof_predictions += oof / len(elastic_configs)
    test_predictions += test_pred / len(elastic_configs)


# ================================
# Final diagnostics
# ================================

print("\nFinal Ensemble Performance")
metric_diagnostics(y, oof_predictions)


# ================================
# Save submission
# ================================

os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    id_col: test[id_col],
    target: test_predictions
})

output_path = "../submissions/exp25_elasticnet_spectral_binning.csv"

submission.to_csv(output_path, index=False, header=False)

print("\nSubmission saved:", output_path)

Original features: 1555
Binned features: 311

Training ElasticNet alpha=0.0005, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.178e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.118e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 13.705683203370018
MAE: 8.911385055810534
RMSLE: 0.4650901865987863
NRMSE (mean): 0.2744557894854056
NRMSE (range): 0.04603219873821931

Training ElasticNet alpha=0.001, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.221e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.150e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 14.200237105775537
MAE: 9.332720333098848
RMSLE: 0.4679028890471502
NRMSE (mean): 0.284359212737916
NRMSE (range): 0.04769321797998138

Training ElasticNet alpha=0.002, l1_ratio=0.9


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.133e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.071e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 15.230933917058737
MAE: 10.17197040715353
RMSLE: 0.5592319373196647
NRMSE (mean): 0.3049988775297641
NRMSE (range): 0.0511549381840622

Training ElasticNet alpha=0.001, l1_ratio=0.8


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.297e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.218e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Metric diagnostics
------------------
RMSE: 15.129405136495425
MAE: 10.088883775277768
RMSLE: 0.5364120060635019
NRMSE (mean): 0.3029657675263064
NRMSE (range): 0.050813941465023035

Final Ensemble Performance

Metric diagnostics
------------------
RMSE: 14.387763431403819
MAE: 9.451811887054747
RMSLE: 0.48987578744352644
NRMSE (mean): 0.2881144203394569
NRMSE (range): 0.04832304788060585

Submission saved: ../submissions/exp25_elasticnet_spectral_binning.csv


/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.200e+05, tolerance: 2.606e+02
  model = cd_fast.enet_coordinate_descent(
